# 第18章　機械学習の学び方いろいろ ― 教師あり・強化・ゼロショット

**『医療診断支援AI開発　第1巻　入門編 ― 動かして、自分で作る（第1巻　入門編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-intro

## 実装 ― scikit-learnで数行

In [ ]:
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import GroupKFold

# X: 患者×特徴量の表（年齢・検査値・腫瘍径…）, y: 正解, groups: 患者ID
clf = RandomForestClassifier(
    n_estimators=500,          # 木の本数。多いほど安定するが、頭打ちになる
    max_features="sqrt",       # 各ノードで見る特徴量の数（分類の既定）
    min_samples_leaf=5,        # 葉に最低何件残すか。小さいほど木が深く、過学習しやすい
    class_weight="balanced",   # 不均衡なら効く（第31章の指標とあわせて使う）
    random_state=0, n_jobs=-1)

# 患者単位で分ける（同じ患者が学習と検証にまたがらないようにする。第32章の鉄則）
cv = GroupKFold(n_splits=5)
for tr, va in cv.split(X, y, groups):
    clf.fit(X[tr], y[tr])
    prob = clf.predict_proba(X[va])[:, 1]   # 陽性確率。閾値は評価の章の作法で決める

# 回帰なら、指標が分散に変わるだけで、書き方は同じ
reg = RandomForestRegressor(n_estimators=500, min_samples_leaf=5,
                            random_state=0, n_jobs=-1)